# 18 · Checkpoint: a streaming CLI chatbot with cost
Level L2 · Part 2 · core · ~90 min · builds on 06, 08, 11, 12, 13, 14, 17

## Goal
Build Scout chat v1: a command-line chatbot on a real model that keeps the history in a list, streams
each reply, prints each turn's cost, retries rate limits safely and never sends more than its window.
Then pass the exit test: show, with printed output, why it forgets when the history is not resent, and
what happens when the history outgrows the window.

## The idea
Lesson 06 built this loop over a fake model. Part 2 gave you every piece needed for a real one: roles and
a resent history (lesson 08), streaming (11), usage and cost (12), stop reasons and errors (13), retries
(14) and a pinned model with a price table (17). This checkpoint joins them into one program.
It also reuses lesson 05's `word_tokens` (for the fake model's token counts) and lesson 07's Gemini client.

The joins are where bugs live. A stream only fails when you read its first chunk. Usage only arrives in
the last chunk. A failed turn must not leave a half-finished turn in the history.

Analogy: you have built an engine, wheels and brakes on separate benches. Now you bolt them together and
take the car for a drive.

## Picture
```mermaid
flowchart LR
    U["you> line"] --> H[("history list")]
    H --> F["fit_to_window<br/>(count_tokens, drop oldest pairs)"]
    F --> O["open_stream<br/>(retry 429/5xx before 1st chunk)"]
    O --> P["print chunks as they arrive"]
    P --> L["last chunk: finish reason + usage"]
    L --> C["Ledger: $ this turn, $ so far"]
    L -->|"ok or cut"| H
```

## Requirements
Your chatbot must:
- [ ] keep one `history` list that starts with the system message and resend it every turn (with `resend_history=False`, only the system message and the newest line);
- [ ] count each request's tokens and drop the **oldest user/assistant pairs** until it fits `window - max_output`, with a warning;
- [ ] stream the reply, printing each chunk as it arrives;
- [ ] retry 408, 429 and 5xx with jittered backoff, only **before** the first chunk is shown; never retry a 400;
- [ ] read the finish reason and usage from the **last** chunk and print `[in, out, status] $ this turn, $ so far`;
- [ ] survive a failed turn: print the error, keep the history valid, carry on;
- [ ] understand `/cost` and `/quit`, and run as a real program (`python scout_chat.py`).

Done when `check_scout_chat(chat_loop)` prints `all checks passed`, and the real model remembers with
history, forgets without it, and forgets again when the window is too small.

## Build it
### Starter code: the recorder and lesson 07's client
Every model call is recorded once to `../data/llm_cassettes/18_checkpoint-cli-chatbot.yaml` and replayed
after that, so this notebook runs with no key.

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("18_checkpoint-cli-chatbot")

import os, random, re, time
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv("../.env")
MODEL = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite")
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)
print("model:", MODEL)

LLM mode: replay (18_checkpoint-cli-chatbot.yaml)


model: gemini-3.5-flash-lite


### Starter code: Scout's notes and lesson 08's message helpers
The same notes and system message as lesson 12, and lesson 08's `to_gemini`, which converts the
vendor-neutral `{"role", "text"}` list to Gemini's `Content` objects at the edge.

In [2]:
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
NOTES_TEXT = "\n".join(f"Note {n['id']}: {n['text']}" for n in NOTES)
SYSTEM_TEXT = ("You are Scout, a research assistant. Answer only from the user's notes below and this chat. "
               "If the notes do not cover it, say so in one sentence. Keep answers to one sentence.\n\n" + NOTES_TEXT)
SYSTEM = {"role": "system", "text": SYSTEM_TEXT}
GEMINI_ROLES = {"user": "user", "assistant": "model"}

def to_gemini(messages: list[dict]) -> tuple[str, list[types.Content]]:
    contents = [types.Content(role=GEMINI_ROLES[m["role"]], parts=[types.Part(text=m["text"])])
                for m in messages[1:]]
    return messages[0]["text"], contents

print(len(NOTES), "notes |", len(to_gemini([SYSTEM, {"role": "user", "text": "Hi"}])[1]), "content")

6 notes | 1 content


### Starter code: cost (lessons 12 and 17)
Lesson 12's `usage_tokens` (thinking counts as output) and `Ledger`, with the dated price table.

In [3]:
PRICES_PER_MTOK = {"gemini-3.5-flash-lite": {"input": 0.30, "output": 2.50}}   # USD, checked 2026-10-06

def cost_usd(model: str, tokens_in: int, tokens_out: int) -> float:
    p = PRICES_PER_MTOK[model]
    return (tokens_in * p["input"] + tokens_out * p["output"]) / 1_000_000

def usage_tokens(resp) -> tuple[int, int]:
    u = resp.usage_metadata
    return u.prompt_token_count or 0, (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)

class Ledger:
    def __init__(self, model: str):
        self.model, self.turns = model, []          # one (input, output, cost) tuple per turn

    def add(self, tokens_in: int, tokens_out: int) -> float:
        c = cost_usd(self.model, tokens_in, tokens_out)
        self.turns.append((tokens_in, tokens_out, c))
        return c

    @property
    def total(self) -> float:
        return sum(t[2] for t in self.turns)

print(f"${cost_usd(MODEL, 1_000, 200):.4f} for 1,000 in + 200 out")

$0.0008 for 1,000 in + 200 out


### Starter code: stop reasons and backoff (lessons 13 and 14)
A stream's status depends on the last chunk's finish reason and on the **joined** text, since the last
chunk usually has no text of its own. `backoff_delay` is lesson 14's full jitter.

In [4]:
BLOCK_REASONS = {"SAFETY", "PROHIBITED_CONTENT", "BLOCKLIST", "SPII", "RECITATION"}
RETRYABLE = {408, 429, 500, 502, 503, 504}

def end_status(last, text: str) -> str:
    if last.prompt_feedback and last.prompt_feedback.block_reason or not last.candidates:
        return "blocked"
    reason = last.candidates[0].finish_reason.value if last.candidates[0].finish_reason else None
    if reason == "MAX_TOKENS":
        return "cut"
    if reason in BLOCK_REASONS:
        return "blocked"
    return "ok" if reason == "STOP" and text else "empty"

def backoff_delay(attempt: int, base: float = 0.5, cap: float = 8.0, rng=random) -> float:
    return rng.uniform(0, min(cap, base * 2 ** attempt))

print("retry on:", sorted(RETRYABLE))

retry on: [408, 429, 500, 502, 503, 504]


### Starter code: two interchangeable backends
Your loop talks to a **backend** with two methods: `count(system, contents)` and `stream(system,
contents, max_output)`. `GeminiBackend` calls the real model. Like the SDK's, its `stream` is a
generator: nothing is sent until you ask for the first chunk.

In [5]:
class GeminiBackend:
    def count(self, system: str, contents: list) -> int:
        cfg = types.CountTokensConfig(system_instruction=system)
        return client.models.count_tokens(model=MODEL, contents=contents, config=cfg).total_tokens

    def stream(self, system: str, contents: list, max_output: int):
        cfg = types.GenerateContentConfig(system_instruction=system, max_output_tokens=max_output,
                                          automatic_function_calling=NO_AFC)
        return client.models.generate_content_stream(model=MODEL, contents=contents, config=cfg)

print("GeminiBackend ready")

GeminiBackend ready


`FakeBackend` is lesson 06's scripted, stateless model, now speaking Gemini's shapes: it yields real
`GenerateContentResponse` chunks, puts usage only in the last one, and can fail on purpose with the
errors Gemini raises (a 429, or a 400 when the input is over its window). First, its chunks and token counter.

In [6]:
def word_tokens(text: str) -> int:                    # lesson 05's toy counter
    return len(re.findall(r"[\w']+|[.,!?]", text))

def chunk(text=None, finish=None, tokens_in=0, tokens_out=0):
    parts = [types.Part(text=text)] if text else []
    usage = types.GenerateContentResponseUsageMetadata(
        prompt_token_count=tokens_in, candidates_token_count=tokens_out) if finish else None
    cand = types.Candidate(content=types.Content(role="model", parts=parts), finish_reason=finish)
    return types.GenerateContentResponse(candidates=[cand], usage_metadata=usage)

def api_error(code: int, status: str, message: str) -> errors.APIError:
    cls = errors.ClientError if code < 500 else errors.ServerError
    return cls(code, {"error": {"code": code, "status": status, "message": message}})

print(chunk("Hello").text, "|", chunk(finish="STOP", tokens_in=5, tokens_out=1).text)

Hello | None


Its replies follow lesson 06's rules: acknowledge "My project is ...", answer "my project" from an
earlier user turn if one was sent, and otherwise quote the best-matching note.

In [7]:
STOPWORDS = {"the", "what", "which", "how", "are", "was", "tell", "about", "me", "is", "a", "on", "in"}

def scripted_reply(contents: list) -> str:
    texts = [c.parts[0].text for c in contents if c.role == "user"]
    user = texts[-1]
    if "my project is" in user.lower():
        return "Got it. I will keep your project in mind."
    if "my project" in user.lower():
        said = [t for t in texts[:-1] if "My project is" in t]
        return ("Your project is" + said[0].split("My project is", 1)[1]) if said \
            else "I don't know what your project is. Could you tell me?"
    words = {w.lower() for w in re.findall(r"[\w']+", user)} - STOPWORDS
    score = lambda n: len(words & {w.lower() for w in re.findall(r"[\w']+", n["text"])})
    best = max(NOTES, key=score)
    return f"From note {best['id']}: {best['text']}" if score(best) else "None of your notes cover that."

Now the backend itself. `stream` yields three words per chunk, then a last chunk with the finish
reason and usage, exactly like Gemini.

In [8]:
class FakeBackend:
    def __init__(self, window: int = 1_000_000, fail_first: int = 0, fail_code: int = 429):
        self.window, self.fail_left, self.fail_code, self.streams = window, fail_first, fail_code, 0

    def count(self, system: str, contents: list) -> int:
        return word_tokens(system) + sum(1 + word_tokens(c.parts[0].text) for c in contents)

    def stream(self, system: str, contents: list, max_output: int):
        self.streams += 1
        n_in = self.count(system, contents)
        if self.fail_left:
            self.fail_left -= 1
            raise api_error(self.fail_code, "RESOURCE_EXHAUSTED" if self.fail_code == 429
                            else "INVALID_ARGUMENT", "Fake failure, as requested.")
        if n_in > self.window:
            raise api_error(400, "INVALID_ARGUMENT", f"The input token count ({n_in}) exceeds "
                            f"the maximum number of tokens allowed ({self.window}).")
        words = scripted_reply(contents).split(" ")[:max_output]
        for i in range(0, len(words), 3):
            yield chunk(" ".join(words[i:i + 3]) + " ")
        yield chunk(finish="STOP", tokens_in=n_in, tokens_out=len(words))

fake = FakeBackend()
print([c.text for c in fake.stream(SYSTEM_TEXT, to_gemini([SYSTEM, {"role": "user", "text": "Saturn's rings?"}])[1], 40)])

['From note 6: ', "Saturn's rings are ", 'mostly made of ', 'ice. ', None]


Four chunks: three of text and a last one with no text (`None`), which holds the usage. The acceptance
test below feeds a scripted conversation to your `chat_loop` with fake backends that behave well, fail
twice with 429, reject the first call with a 400, and have a tiny window.

In [9]:
import contextlib, io

WINDOW, MAX_OUTPUT = 2_000, 120          # Scout's own request budget; the model's real window is far larger
TEST_INPUTS = [
    "Hi Scout. My project is a school talk about the moons of Mars.",
    "How long is a day on Mars?",
    "Which rover landed on Mars in 2021?",
    "/cost",
    "What did I tell you about my project?",
    "/quit",
    "this line comes after /quit and must never be sent",
]

def run_quietly(fn):
    out = io.StringIO()
    with contextlib.redirect_stdout(out):
        result = fn()
    return result, out.getvalue()

The checks themselves. Each one runs your loop on a fresh fake backend and inspects what it returned
and printed.

In [10]:
def check_scout_chat(chat_loop) -> None:
    (hist, ledger), out = run_quietly(lambda: chat_loop(FakeBackend(), TEST_INPUTS))
    assert hist[0] == SYSTEM and len(hist) == 9, f"expected system + 4 pairs, got {len(hist)} messages"
    assert hist[-1]["text"].startswith("Your project is"), "with history Scout should remember"
    assert len(ledger.turns) == 4 and "so far" in out and "never be sent" not in out
    (hist, _), _ = run_quietly(lambda: chat_loop(FakeBackend(), TEST_INPUTS, resend_history=False))
    assert "don't know" in hist[-1]["text"], "without history Scout should forget"
    flaky = FakeBackend(fail_first=2)
    (hist, _), out = run_quietly(lambda: chat_loop(flaky, TEST_INPUTS))
    assert len(hist) == 9 and flaky.streams == 6 and "retry" in out, "two 429s should be retried"
    bad = FakeBackend(fail_first=1, fail_code=400)
    (hist, _), out = run_quietly(lambda: chat_loop(bad, TEST_INPUTS))
    assert bad.streams == 4 and len(hist) == 7 and "400" in out, "a 400 is reported once, never retried"
    small = FakeBackend().count(SYSTEM_TEXT, []) + MAX_OUTPUT + 40
    (hist, _), out = run_quietly(lambda: chat_loop(FakeBackend(window=small), TEST_INPUTS, window=small))
    assert "dropped" in out and "don't know" in hist[-1]["text"], "a small window should trim and warn"
    print("all checks passed")

### Your turn
Write `chat_loop(backend, inputs, resend_history=True, window=WINDOW, max_output=MAX_OUTPUT)` that
returns `(history, ledger)`, and run `check_scout_chat(chat_loop)`. Then run it on `GeminiBackend()`,
and write `scout_chat.py`, a program that reads from the keyboard.

In [11]:
# your code here

## Reference solution
Try it yourself first. The steps below are one way to meet every requirement.

### R1. Fit the request to the window
Count the whole request with the backend (system message included, lesson 12). While it is over
`window - max_output`, drop the oldest user/assistant **pair**, so the request still starts with a user
turn. Only the request is trimmed; the stored history keeps everything.

In [12]:
def fit_to_window(backend, messages: list[dict], window: int, max_output: int):
    system, rest = messages[0], messages[1:]
    while True:
        text, contents = to_gemini([system] + rest)
        n_in = backend.count(text, contents)
        if n_in <= window - max_output or len(rest) <= 1:
            return [system] + rest, n_in, len(messages) - 1 - len(rest)
        rest = rest[2:]                                   # oldest user + assistant pair

small = FakeBackend().count(SYSTEM_TEXT, []) + MAX_OUTPUT + 40
long_chat = [SYSTEM] + [{"role": r, "text": "Mars has two small moons."} for r in ["user", "assistant"] * 4]
kept, n_in, dropped = fit_to_window(FakeBackend(), long_chat + [{"role": "user", "text": "And?"}], small, MAX_OUTPUT)
print(f"window {small}: kept {len(kept)} messages ({n_in} tokens), dropped {dropped}")

window 269: kept 6 messages (140 tokens), dropped 4


### R2. Open the stream, retrying only before the first chunk
The SDK's stream is a generator, so a 429 appears when you ask for the **first** chunk. Ask for it
inside the `try`. Once a chunk is on screen, a retry would print the start of the reply twice, so
`open_stream` hands the rest of the stream back without retrying.

In [13]:
RNG = random.Random(18)

def open_stream(backend, system: str, contents: list, max_output: int, max_attempts: int = 4):
    for attempt in range(max_attempts):
        try:
            stream = backend.stream(system, contents, max_output)
            return next(stream), stream
        except errors.APIError as e:
            if e.code not in RETRYABLE or attempt == max_attempts - 1:
                raise
            wait = backoff_delay(attempt, rng=RNG)
            print(f"  ! {e.code} {e.status}: retry {attempt + 1} in {wait:.2f}s")
            time.sleep(wait)

flaky = FakeBackend(fail_first=2)
first, rest = open_stream(flaky, *to_gemini([SYSTEM, {"role": "user", "text": "Saturn's rings?"}]), MAX_OUTPUT)
print("first chunk:", repr(first.text), "| calls:", flaky.streams)

  ! 429 RESOURCE_EXHAUSTED: retry 1 in 0.09s


  ! 429 RESOURCE_EXHAUSTED: retry 2 in 0.66s


first chunk: 'From note 6: ' | calls: 3


### R3. One turn: stream, then account
Print each chunk as it arrives. When the stream ends, join the text, take status and usage from the last
chunk, and charge the ledger. A failed or blocked turn removes its user line, so the history stays a
clean list of user/assistant pairs.

In [14]:
def chat_turn(backend, history, user_text, ledger, resend_history=True, window=WINDOW, max_output=MAX_OUTPUT):
    history.append({"role": "user", "text": user_text})
    to_send = history if resend_history else [history[0], history[-1]]
    try:
        trimmed, _, dropped = fit_to_window(backend, to_send, window, max_output)
        if dropped:
            print(f"  ! dropped {dropped} old messages to fit the {window}-token window")
        first, stream = open_stream(backend, *to_gemini(trimmed), max_output)
        chunks = [first]
        print("scout> " + (first.text or ""), end="", flush=True)
        for ch in stream:
            chunks.append(ch)
            print(ch.text or "", end="", flush=True)
    except errors.APIError as e:
        history.pop()
        return print(f"  ! {e.code} {e.status}: {e.message[:70]} (turn not saved)")
    text = "".join(ch.text or "" for ch in chunks).strip()
    status, (n_in, n_out) = end_status(chunks[-1], text), usage_tokens(chunks[-1])
    cost = ledger.add(n_in, n_out)
    print(f"\n       [{n_in} in, {n_out} out, {status}] ${cost:.5f} this turn, ${ledger.total:.5f} so far")
    if status in ("ok", "cut"):
        history.append({"role": "assistant", "text": text})
    else:
        history.pop()

### R4. The loop and the acceptance checks
The loop reads lines, handles the two commands and calls `chat_turn`. Then run the checks.

In [15]:
def chat_loop(backend, inputs, resend_history=True, window=WINDOW, max_output=MAX_OUTPUT):
    history, ledger = [SYSTEM], Ledger(MODEL)
    for line in inputs:
        if line.strip() == "/quit":
            print("scout> Bye.")
            break
        if line.strip() == "/cost":
            print(f"  {len(ledger.turns)} turns, ${ledger.total:.5f} so far")
            continue
        print(f"you>   {line}")
        chat_turn(backend, history, line, ledger, resend_history, window, max_output)
    return history, ledger

check_scout_chat(chat_loop)

all checks passed


To see what the checks saw, here is the 400 case: the first turn fails once (never retried), is not
saved, and the chat carries on.

In [16]:
history_bad, _ = chat_loop(FakeBackend(fail_first=1, fail_code=400), TEST_INPUTS[:2] + ["/quit"])
print(len(history_bad), "messages:", [m["role"] for m in history_bad])

you>   Hi Scout. My project is a school talk about the moons of Mars.
  ! 400 INVALID_ARGUMENT: Fake failure, as requested. (turn not saved)
you>   How long is a day on Mars?
scout> From note 3: 

A day on 

Mars lasts about 

24 hours and 

39 minutes. 


       [118 in, 14 out, ok] $0.00007 this turn, $0.00007 so far
scout> Bye.
3 messages: ['system', 'user', 'assistant']


### R5. Scout chat v1 on the real model
Same loop, real backend. Watch the input tokens grow each turn, as in lesson 12.

In [17]:
import json, pathlib
RECORDED = pathlib.Path("../data/recorded/18.json")
runs = {}

def keep(name, history, ledger, warnings=()):        # saved so the lesson still runs without a key
    runs[name] = {"answers": [m["text"] for m in history if m["role"] == "assistant"], "turns": ledger.turns,
                  "warnings": list(warnings)}
    RECORDED.write_text(json.dumps(runs, indent=1))

history_full, ledger_full = chat_loop(GeminiBackend(), TEST_INPUTS)
keep("full", history_full, ledger_full)

you>   Hi Scout. My project is a school talk about the moons of Mars.
scout> Mars

 has two small moons, Phobos and Deimos.


       [156 in, 12 out, ok] $0.00008 this turn, $0.00008 so far
you>   How long is a day on Mars?
scout> A day

 on Mars lasts about 24 hours and 39 minutes.


       [176 in, 16 out, ok] $0.00009 this turn, $0.00017 so far
you>   Which rover landed on Mars in 2021?
scout> The rover

 Perseverance landed on Mars in February 2021.


       [204 in, 14 out, ok] $0.00010 this turn, $0.00027 so far
  3 turns, $0.00027 so far
you>   What did I tell you about my project?
scout> Your project

 is a school talk about the moons of Mars.


       [227 in, 12 out, ok] $0.00010 this turn, $0.00036 so far
scout> Bye.


If you skipped the cell above (no key and no recording), this loads its recorded output.

In [18]:
import json, pathlib
RECORDED = pathlib.Path("../data/recorded/18.json")
if "history_full" not in globals():                  # recorded output of the real chat
    runs = json.loads(RECORDED.read_text())
    print("recorded answers:", *runs["full"]["answers"], sep="\n  ")
print("input tokens per turn:", [t[0] for t in runs["full"]["turns"]])
print(f"chat total: ${sum(t[2] for t in runs['full']['turns']):.5f}")

input tokens per turn: [156, 176, 204, 227]
chat total: $0.00036


Turn 4 was answered from turn 1, because turn 1 was in the request. The input count grew every turn,
since each request carries the whole chat so far. (Turn 1 got a Mars fact instead of an acknowledgement:
the system message says "answer from the notes". Better prompts are lesson 19.)

### R6. Exit test, part 1: why it forgets
Same model, same lines, but each request now carries only the system message and the newest line. The
loop still stores the history; it just does not send it.

In [19]:
history_short, ledger_short = chat_loop(GeminiBackend(), TEST_INPUTS, resend_history=False)
keep("no_history", history_short, ledger_short)

you>   Hi Scout. My project is a school talk about the moons of Mars.
scout> Mars

 has two small moons, Phobos and Deimos.


       [156 in, 12 out, ok] $0.00008 this turn, $0.00008 so far
you>   How long is a day on Mars?
scout> A day

 on Mars lasts about 24 hours and 39 minutes.


       [149 in, 16 out, ok] $0.00008 this turn, $0.00016 so far
you>   Which rover landed on Mars in 2021?
scout> The rover

 Perseverance landed on Mars in February 2021.


       [153 in, 14 out, ok] $0.00008 this turn, $0.00024 so far
  3 turns, $0.00024 so far
you>   What did I tell you about my project?
scout> My notes

 do not contain any information about your project.


       [150 in, 11 out, ok] $0.00007 this turn, $0.00031 so far
scout> Bye.


Compare the last answer and the input tokens of both runs.

In [20]:
if "history_short" not in globals():
    runs = json.loads(RECORDED.read_text())          # recorded output
for name in ("full", "no_history"):
    print(f"{name:10} inputs {[t[0] for t in runs[name]['turns']]} -> {runs[name]['answers'][-1][:70]}")

full       inputs [156, 176, 204, 227] -> Your project is a school talk about the moons of Mars.
no_history inputs [156, 149, 153, 150] -> My notes do not contain any information about your project.


Without the history, the input stays flat (system message plus one line), and the model cannot know the
project: it was never in the request. The model is stateless (lesson 08); the list in **your** program
is the chat's only memory. Stop sending it and the memory is gone, even though it is still in `history`.

### R7. Exit test, part 2: when the history outgrows the window
First, what happens if you do **not** trim. The fake backend has a small window and raises the same
400 that Gemini returns for an over-long input (RESEARCH section 15). Count the request first:

In [21]:
tiny = FakeBackend(window=small)
long_request = long_chat[1:] * 6 + [{"role": "user", "text": "What is my project about?"}]
system_text, contents = to_gemini([SYSTEM] + long_request)
print(tiny.count(system_text, contents), "tokens in the request, window", small)

452 tokens in the request, window 269


The request is well over the window. Now send it untrimmed through `open_stream`.

In [22]:
try:
    open_stream(tiny, system_text, contents, MAX_OUTPUT)
except errors.APIError as e:
    print(type(e).__name__, e.code, e.status, "|", e.message)

ClientError 400 INVALID_ARGUMENT | The input token count (452) exceeds the maximum number of tokens allowed (269).


A 400, not a 429: it is not retried (no "retry" line above), and every later turn is longer still, so
an untrimmed chat is stuck for good. With trimming, the real model keeps working but drops the oldest
turns. Here the window is set just large enough for the system message and a few short turns.

In [23]:
import contextlib, io
SMALL_WINDOW = 280
with contextlib.redirect_stdout(io.StringIO()) as out:   # capture the printout to keep its warnings
    history_small, ledger_small = chat_loop(GeminiBackend(), TEST_INPUTS, window=SMALL_WINDOW, max_output=60)
print(out.getvalue())
keep("small_window", history_small, ledger_small, [l.strip() for l in out.getvalue().splitlines() if "dropped" in l])

you>   Hi Scout. My project is a school talk about the moons of Mars.
scout> Mars has two small moons, Phobos and Deimos.
       [156 in, 12 out, ok] $0.00008 this turn, $0.00008 so far
you>   How long is a day on Mars?
scout> A day on Mars lasts about 24 hours and 39 minutes.
       [176 in, 16 out, ok] $0.00009 this turn, $0.00017 so far
you>   Which rover landed on Mars in 2021?
scout> The rover Perseverance landed on Mars in February 2021.
       [204 in, 14 out, ok] $0.00010 this turn, $0.00027 so far
  3 turns, $0.00027 so far
you>   What did I tell you about my project?
  ! dropped 2 old messages to fit the 280-token window
scout> The notes do not cover your project.
       [200 in, 8 out, ok] $0.00008 this turn, $0.00035 so far
scout> Bye.



The three runs side by side.

In [24]:
if "history_small" not in globals():
    runs = json.loads(RECORDED.read_text())          # recorded output
for name, run in runs.items():
    total = sum(t[2] for t in run["turns"])
    print(f"{name:12} ${total:.5f}  inputs {[t[0] for t in run['turns']]}\n  -> {run['answers'][-1][:80]}")
print("small_window warnings:", runs["small_window"]["warnings"])

full         $0.00036  inputs [156, 176, 204, 227]
  -> Your project is a school talk about the moons of Mars.
no_history   $0.00031  inputs [156, 149, 153, 150]
  -> My notes do not contain any information about your project.
small_window $0.00035  inputs [156, 176, 204, 200]
  -> The notes do not cover your project.
small_window warnings: ['! dropped 2 old messages to fit the 280-token window']


With the small window the input stops growing, and the warning above shows why: on the last turn
`fit_to_window` dropped the oldest pair (2 messages), the one with "My project is ...", so Scout forgot
again. This time not because we skipped resending, but because the history no longer fit. That is the whole exit test: **forgetting comes from what is in the request**.

### R8. A real command-line program
The same steps as a script you can run in a terminal. It talks to Gemini directly and stores the system
instruction separately, so its history holds only user/assistant pairs.

In [25]:
from pathlib import Path
Path("../apps/18").mkdir(parents=True, exist_ok=True)
print("ready to write ../apps/18/scout_chat.py")

ready to write ../apps/18/scout_chat.py


Under 80 lines: client and prices, `fit` (count, drop pairs), `open_stream` (retry before the first
chunk), and `main()`. Set `SCOUT_WINDOW` to try a smaller window. The two lines under `SCOUT_CASSETTE` only let this course record and replay it.

In [26]:
%%writefile ../apps/18/scout_chat.py
"""Scout chat v1: python scout_chat.py [--no-history]   (needs GOOGLE_CLOUD_API_KEY)"""
import os, random, sys, time
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types
if os.environ.get("SCOUT_CASSETTE"):           # only so the course can record and replay this script
    import llm_replay; llm_replay.start(os.environ["SCOUT_CASSETTE"])
load_dotenv("../.env")
MODEL, MAX_OUTPUT = os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite"), 120
PRICE_IN, PRICE_OUT = 0.30, 2.50                # USD per million tokens, checked 2026-10-06
WINDOW = int(os.environ.get("SCOUT_WINDOW", 2000))   # tokens per request, system message included
NOTES = ["Mars has two small moons, Phobos and Deimos.", "Venus spins backwards compared with most planets.",
         "A day on Mars lasts about 24 hours and 39 minutes.", "Jupiter is the largest planet in the solar system.",
         "The rover Perseverance landed on Mars in February 2021.", "Saturn's rings are mostly made of ice."]
SYSTEM = ("You are Scout, a research assistant. Answer only from the user's notes below and this chat. "
          "If the notes do not cover it, say so in one sentence. Keep answers to one sentence.\n\n"
          + "\n".join(f"Note {i}: {t}" for i, t in enumerate(NOTES, 1)))
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed")

def contents_of(history):
    return [types.Content(role={"user": "user", "assistant": "model"}[m["role"]], parts=[types.Part(text=m["text"])])
            for m in history]

def fit(history, window):                       # drop oldest pairs until it fits window - MAX_OUTPUT
    config, dropped = types.CountTokensConfig(system_instruction=SYSTEM), 0
    while len(history) > 1 and client.models.count_tokens(
            model=MODEL, contents=contents_of(history), config=config).total_tokens > window - MAX_OUTPUT:
        history, dropped = history[2:], dropped + 2
    return history, dropped

def open_stream(contents, attempts=4):          # retry 408/429/5xx only before anything is shown
    config = types.GenerateContentConfig(system_instruction=SYSTEM, max_output_tokens=MAX_OUTPUT,
                                         automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True))
    for attempt in range(attempts):
        try:
            stream = client.models.generate_content_stream(model=MODEL, contents=contents, config=config)
            return next(stream), stream
        except errors.APIError as e:
            if e.code not in {408, 429, 500, 502, 503, 504} or attempt == attempts - 1:
                raise
            print(f"  ! {e.code} {e.status}: retrying")
            time.sleep(random.uniform(0, min(8.0, 0.5 * 2 ** attempt)))

def main():
    no_history, history, total, piped = "--no-history" in sys.argv, [], 0.0, not sys.stdin.isatty()
    while True:
        try:
            line = input("" if piped else "you> ").strip()
        except EOFError:                         # Ctrl-D, or the end of a pipe
            break
        print("you>   " + line) if piped else None
        if line == "/quit":
            break
        if line == "/cost":
            print(f"  {len(history) // 2} turns, ${total:.5f} so far"); continue
        history.append({"role": "user", "text": line})
        try:
            to_send, dropped = fit(history[-1:] if no_history else history, WINDOW)
            print(f"  ! dropped {dropped} old messages to fit the window") if dropped else None
            first, stream = open_stream(contents_of(to_send))
            chunks = [first]
            print("scout> " + (first.text or ""), end="", flush=True)
            for ch in stream:
                chunks.append(ch)
                print(ch.text or "", end="", flush=True)
        except errors.APIError as e:
            history.pop()
            print(f"  ! {e.code} {e.status}: {e.message[:70]} (turn not saved)")
            continue
        last, u = chunks[-1], chunks[-1].usage_metadata
        reason = last.candidates[0].finish_reason.value if last.candidates else "BLOCKED"
        n_in, n_out = u.prompt_token_count or 0, (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)
        total += (cost := (n_in * PRICE_IN + n_out * PRICE_OUT) / 1_000_000)
        history.append({"role": "assistant", "text": "".join(c.text or "" for c in chunks).strip()})
        print(f"\n       [{n_in} in, {n_out} out, {reason}] ${cost:.5f} this turn, ${total:.5f} so far")
    print(f"scout> Bye. {len(history) // 2} turns, ${total:.5f}")

if __name__ == "__main__":
    main()

Overwriting ../apps/18/scout_chat.py


Run it the way a terminal would, piping the test lines into it. In your own terminal, run
`python ../apps/18/scout_chat.py` from the `notebooks` folder and type; `/quit` or Ctrl-D ends it.

In [27]:
import subprocess

session = "\n".join(TEST_INPUTS[:6]) + "\n"                 # four questions, /cost, /quit
env = os.environ | {"SCOUT_CASSETTE": "18_scout_cli", "PYTHONPATH": ".."}
run = subprocess.run([sys.executable, "../apps/18/scout_chat.py"], input=session, env=env,
                     capture_output=True, text=True, timeout=120)
print(run.stdout[-900:] or run.stderr[-900:])

LLM mode: replay (18_scout_cli.yaml)
you>   Hi Scout. My project is a school talk about the moons of Mars.
scout> Mars has two small moons, Phobos and Deimos.
       [156 in, 12 out, STOP] $0.00008 this turn, $0.00008 so far
you>   How long is a day on Mars?
scout> A day on Mars lasts about 24 hours and 39 minutes.
       [176 in, 16 out, STOP] $0.00009 this turn, $0.00017 so far
you>   Which rover landed on Mars in 2021?
scout> The rover Perseverance landed on Mars in February 2021.
       [204 in, 14 out, STOP] $0.00010 this turn, $0.00027 so far
you>   /cost
  3 turns, $0.00027 so far
you>   What did I tell you about my project?
scout> Your project is a school talk about the moons of Mars.
       [227 in, 12 out, STOP] $0.00010 this turn, $0.00036 so far
you>   /quit
scout> Bye. 4 turns, $0.00036



The program streams, answers `/cost`, prints the cost of each turn and remembers the project, like the
notebook. One difference: it prints Gemini's raw finish reason (`STOP`) where the notebook's loop prints
`ok`, `cut` or `blocked`.

## At work
- **Every call is logged with its usage, cost, stop reason and your own call id**, so "why was yesterday's
  bill high?" and "which turn was cut?" have answers. Production chat services do this per user and per
  feature. → lesson 49
- **Long chats are trimmed, summarised or cached, never sent whole forever.** Dropping the oldest turns
  (as here) loses facts like the user's project; compaction keeps a summary instead, and prompt caching
  bills the repeated prefix at about 10% of the input price. → lessons 32 and 41
- **Spending is capped before the call**, using a counted input and the `max_output_tokens` worst case,
  plus provider-side monthly limits set below the tier cap. → lesson 54
- **A server keeps one history per user and many chats at once**, under shared rate limits. Retries must
  not duplicate side effects once tools write things. → lessons 38 and 39

## What just happened
- `fit_to_window` counted each request with the backend and dropped the oldest pairs; only the request was trimmed, never the stored history.
- `open_stream` retried two fake 429s and returned the first chunk on the third call; a 400 was reported once, never retried, and its turn was not saved.
- `chat_turn` printed chunks as they arrived, took status and usage from the last chunk, and printed `$ this turn, $ so far`.
- On Gemini, the input grew every turn and Scout remembered the project; without resending the history, the input stayed flat and Scout did not know it.
- Over the window, an untrimmed request got a 400 and would never recover; with a small window, trimming dropped the first turn and Scout forgot again.
- `scout_chat.py` ran the same steps as a real program over a pipe.

## Common mistakes
**1. Wrapping only the creation of the stream in `try`.** It looks safe, but the SDK's stream is a
generator: nothing is sent until the first `next()`. The 429 escapes the retry loop.

In [28]:
def open_stream_wrong(backend, system, contents, max_output):
    for attempt in range(4):
        try:
            return backend.stream(system, contents, max_output)    # no request has been made yet
        except errors.APIError:
            time.sleep(backoff_delay(attempt, rng=RNG))

print("defined open_stream_wrong")

defined open_stream_wrong


Use it on a backend that fails once with a 429.

In [29]:
contents = to_gemini([SYSTEM, {"role": "user", "text": "Saturn's rings?"}])[1]
for piece in open_stream_wrong(FakeBackend(fail_first=1), SYSTEM_TEXT, contents, MAX_OUTPUT):
    print(piece.text, end="")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'status': 'RESOURCE_EXHAUSTED', 'message': 'Fake failure, as requested.'}}

Fix: ask for the first chunk inside the `try`, as `open_stream` does (`return next(stream), stream`).

**2. Saving the last chunk as the reply.** The last chunk carries the usage, and usually no text at all.

In [30]:
chunks = list(FakeBackend().stream(SYSTEM_TEXT, contents, MAX_OUTPUT))
reply = chunks[-1].text.strip()

AttributeError: 'NoneType' object has no attribute 'strip'

Fix: join every chunk's text, and read only the usage and the finish reason from the last chunk.

In [31]:
reply = "".join(ch.text or "" for ch in chunks).strip()
print(repr(reply), "|", usage_tokens(chunks[-1]))

"From note 6: Saturn's rings are mostly made of ice." | (113, 10)


## Try it
Add a per-chat spending cap. Write `chat_loop_capped(backend, inputs, budget_usd)`, like `chat_loop`,
that before each turn computes the worst-case cost (counted input plus `MAX_OUTPUT` output tokens,
lesson 12) and, if `ledger.total` plus that would pass `budget_usd`, prints `budget reached` and stops.
Test it on `FakeBackend()` with a budget that allows exactly two turns. No model calls needed.

In [32]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [33]:
llm_replay.stop()

## Key terms
- **backend** — the object your chat loop calls for the model (`count`, `stream`); swapping a real one for a fake one lets the same loop be tested with no key.
- **retry before first chunk** — retrying a streamed call only while nothing has been shown yet; after that, a retry would repeat text the user has already seen.
- **request trimming** — dropping the oldest user/assistant pairs from the request (not from the stored history) until it fits the window minus the reserved output.
- **exit test** — the checkpoint's final demonstration: forgetting comes only from what is in the request, either because the history was not resent or because it was trimmed to fit the window.